# Local Document Reader
## Notebook 02: Embeddings and FAISS Vector Database

### Objective
This notebook turns the processed chunks from Phase 2 into searchable numerical vectors. It uses a local Ollama embedding model, normalizes its vectors, stores them in a FAISS index, and retrieves the most relevant source passages for a natural-language question.


### 1. One-time dependency and model check
Run this command in the project terminal before executing the embedding cells: `venv/bin/python -m pip install faiss-cpu ollama`. Then ensure Ollama is running and the `nomic-embed-text` model is present with `ollama list`; if needed run `ollama pull nomic-embed-text`.

Why: FAISS handles vector search and Ollama generates vectors locally, so no cloud API key is required.

In [1]:
import json
import sys
from pathlib import Path
from typing import Any

import faiss
import numpy as np
import ollama
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VECTOR_DB_DIR = PROJECT_ROOT / 'data' / 'vector_store'
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_MODEL = 'nomic-embed-text'

print('Libraries imported successfully!')
print('Processed chunks:', PROCESSED_DIR)
print('Vector store:', VECTOR_DB_DIR)
print('Embedding model:', EMBEDDING_MODEL)

Libraries imported successfully!
Processed chunks: /Users/apple/Downloads/Assessment/data/processed
Vector store: /Users/apple/Downloads/Assessment/data/vector_store
Embedding model: nomic-embed-text


### 2. Load processed chunks
An embedding only represents the chunk text. The remaining fields — especially source and page number — remain in a parallel metadata list, so a FAISS result can be turned back into a citable document passage.

In [2]:
chunk_files = sorted(PROCESSED_DIR.glob('*_chunks.json'))
if not chunk_files:
    raise FileNotFoundError('No processed chunk JSON files found. Run Notebook 01 first.')

all_chunks: list[dict[str, Any]] = []
for file_path in chunk_files:
    with file_path.open(encoding='utf-8') as file:
        all_chunks.extend(json.load(file))

if not all_chunks:
    raise ValueError('Processed chunk files were empty.')

chunks_df = pd.DataFrame(all_chunks)
print('Processed files found:', len(chunk_files))
print('Total chunks loaded:', len(all_chunks))
display(chunks_df[['chunk_id', 'source', 'page_number', 'character_count', 'text']].head())

Processed files found: 1
Total chunks loaded: 2


,chunk_id,source,page_number,character_count,text
0,client_requirements_page_1_chunk_1,client_requirements.pdf,1,200,Client Requirements Project: School Management...
1,client_requirements_page_2_chunk_1,client_requirements.pdf,2,109,Acceptance Notes The system must provide role-...


### 3. Generate embeddings with task prefixes
An embedding is a numerical vector that represents the meaning of text. Nomic uses `search_document:` for source chunks and `search_query:` for questions. Using the correct prefix keeps document and question embeddings compatible in the same search space.

Expected result: an array with one vector per input text. The model determines the vector dimension; this notebook does not hard-code 768.

In [3]:
def _embedding_values(response: Any) -> list[list[float]]:
    """Support both dict-like and current Ollama client responses."""
    return response['embeddings'] if isinstance(response, dict) else response.embeddings

def generate_embeddings(texts: list[str], model: str = EMBEDDING_MODEL, batch_size: int = 32) -> np.ndarray:
    """Embed document text in batches, preserving input order."""
    if not texts:
        raise ValueError('Input text list is empty.')
    if batch_size <= 0:
        raise ValueError('Batch size must be positive.')

    vectors: list[list[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = ['search_document: ' + text.strip() for text in texts[start:start + batch_size]]
        response = ollama.embed(model=model, input=batch)
        vectors.extend(_embedding_values(response))

    embeddings = np.ascontiguousarray(vectors, dtype=np.float32)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(texts):
        raise ValueError('Ollama returned an unexpected embedding shape.')
    return embeddings

sample_embedding = generate_embeddings(['The School Management System uses PostgreSQL as its database.'])
print('Embedding generated successfully!')
print('Vector shape:', sample_embedding.shape)
print('First 10 values:', sample_embedding[0, :10])

Embedding generated successfully!
Vector shape: (1, 768)
First 10 values: [-0.01020381  0.06212812 -0.15988049 -0.02487035  0.02811188 -0.01008086
  0.00790604 -0.00650888  0.03804073 -0.01547749]


### 4. Normalize and index vectors
FAISS `IndexFlatIP` performs exact inner-product search. If both the document and query vectors have L2 norm 1, inner product equals cosine similarity. Scores closer to 1 imply more similar vector direction; they are not probabilities.

In [4]:
document_embeddings = generate_embeddings([chunk['text'] for chunk in all_chunks])
faiss.normalize_L2(document_embeddings)

vector_norms = np.linalg.norm(document_embeddings, axis=1)
assert np.allclose(vector_norms, 1.0, atol=1e-5)

embedding_dimension = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(document_embeddings)
assert faiss_index.ntotal == len(all_chunks)
print('Embedding matrix shape:', document_embeddings.shape)
print('FAISS index created with vectors:', faiss_index.ntotal)
print('All document vectors are L2 normalized.')

Embedding matrix shape: (2, 768)
FAISS index created with vectors: 2
All document vectors are L2 normalized.


### 5. Semantic search
The search function embeds a question with the query prefix, normalizes it, searches FAISS, and joins returned vector positions to the original chunk metadata. This is how retrieved passages retain their source and PDF page.

In [5]:
def generate_query_embedding(query: str, model: str = EMBEDDING_MODEL) -> np.ndarray:
    if not isinstance(query, str) or not query.strip():
        raise ValueError('Query cannot be empty.')
    response = ollama.embed(model=model, input='search_query: ' + query.strip())
    vector = np.asarray(_embedding_values(response)[0], dtype=np.float32).reshape(1, -1)
    if vector.shape[1] != faiss_index.d:
        raise ValueError('Query and document embedding dimensions differ.')
    faiss.normalize_L2(vector)
    return vector

def search_documents(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    if top_k <= 0:
        raise ValueError('top_k must be positive.')
    scores, indices = faiss_index.search(generate_query_embedding(query), min(top_k, faiss_index.ntotal))
    results = []
    for score, index in zip(scores[0], indices[0]):
        if index < 0:
            continue
        chunk = all_chunks[int(index)]
        results.append({**chunk, 'similarity_score': float(score)})
    return results

def display_search_results(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame([
        {'rank': rank, 'similarity_score': round(item['similarity_score'], 4), 'source': item['source'], 'page': item['page_number'], 'chunk_id': item['chunk_id'], 'text': item['text']}
        for rank, item in enumerate(results, start=1)
    ])

In [6]:
query = 'What database does the application use?'
search_results = search_documents(query, top_k=3)
display(display_search_results(search_results))

for rank, result in enumerate(search_results, start=1):
    print(f'#{rank} | {result["source"]} | page {result["page_number"]} | score {result["similarity_score"]:.4f}')
    print(result['text'])
    print('-' * 70)

,rank,similarity_score,source,page,chunk_id,text
0,1,0.6848,client_requirements.pdf,1,client_requirements_page_1_chunk_1,Client Requirements Project: School Management...
1,2,0.5732,client_requirements.pdf,2,client_requirements_page_2_chunk_1,Acceptance Notes The system must provide role-...


#1 | client_requirements.pdf | page 1 | score 0.6848
Client Requirements Project: School Management System Frontend: React Backend: Python FastAPI Database: PostgreSQL Deployment: AWS Deliverables: Admin dashboard, attendance management, fee management.
----------------------------------------------------------------------
#2 | client_requirements.pdf | page 2 | score 0.5732
Acceptance Notes The system must provide role-based access and maintain auditable attendance and fee records.
----------------------------------------------------------------------


### 6. Persist and verify the vector store
The index and metadata must be saved together in exactly the same order. On restart, the index position returned by FAISS points to the metadata record at the same list position.

In [7]:
index_path = VECTOR_DB_DIR / 'documind.faiss'
metadata_path = VECTOR_DB_DIR / 'metadata.json'
faiss.write_index(faiss_index, str(index_path))
with metadata_path.open('w', encoding='utf-8') as file:
    json.dump(all_chunks, file, indent=2, ensure_ascii=False)

loaded_index = faiss.read_index(str(index_path))
with metadata_path.open(encoding='utf-8') as file:
    loaded_metadata = json.load(file)
assert loaded_index.ntotal == len(loaded_metadata) == len(all_chunks)
print('Vector database saved and reloaded successfully!')
print('Index:', index_path)
print('Metadata:', metadata_path)

Vector database saved and reloaded successfully!
Index: /Users/apple/Downloads/Assessment/data/vector_store/documind.faiss
Metadata: /Users/apple/Downloads/Assessment/data/vector_store/metadata.json


### 7. Baseline retrieval evaluation
This is a small smoke test, not a full retrieval benchmark. It checks that the expected keyword appears somewhere in the top three retrieved chunks. If it fails, inspect the retrieved passages before changing model or similarity settings.

In [8]:
evaluation_queries = [
    {'question': 'Which database is used?', 'expected_keyword': 'PostgreSQL'},
    {'question': 'Which frontend framework is required?', 'expected_keyword': 'React'},
    {'question': 'Which backend framework is used?', 'expected_keyword': 'FastAPI'},
]
evaluation_results = []
for item in evaluation_queries:
    retrieved_text = ' '.join(result['text'] for result in search_documents(item['question'], top_k=3)).lower()
    evaluation_results.append({**item, 'passed': item['expected_keyword'].lower() in retrieved_text})

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df)
print('Keyword retrieval success:', f'{evaluation_df.passed.mean():.0%}')

,question,expected_keyword,passed
0,Which database is used?,PostgreSQL,True
1,Which frontend framework is required?,React,True
2,Which backend framework is used?,FastAPI,True


Keyword retrieval success: 100%
